|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 7. You can guess tokens and check them, store fewer bytes
for each weight, forbid the tokens that break a schema, and cut a model
across ranks. Each feature has a promise: the same distribution, almost the
same output, always valid, the same result. Most tickets in this file are a
promise that the code did not keep, or a promise that nobody made.

Each ticket gives you a **symptom** and some **evidence**. Some of the
evidence is noise. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell.

Do this section after stage 20. This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 7.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

| GPU | Memory | Bandwidth |
|---|---|---|
| A100 SXM 80GB | 80 GB | 2,039 GB/s |
| L40S | 48 GB | 864 GB/s |

| Model | Layers | Attention heads | KV heads | head_dim | hidden | bf16 weights |
|---|---|---|---|---|---|---|
| Qwen3-0.6B | 28 | 16 | 8 | 128 | 1024 | 1.5 GB |
| Qwen3-1.7B | 28 | 16 | 8 | 128 | 2048 | 3.44 GB |
| Llama-3-8B | 32 | 32 | 8 | 128 | 4096 | 16.1 GB |

- Speculative decoding with k draft tokens and an acceptance rate a for
  each token: the expected tokens for each verify step are
  `(1 - a^(k+1)) / (1 - a)`.
- FP8 e4m3: the largest value is 448. PyTorch clamps a larger value to 448.
- int8: 256 levels. With a scale s, the step between two levels is s.
- One all-reduce of a few KB over PCIe costs about 25 µs, most of it
  latency.

# Ticket 1: speculation that slows the chat

**Severity:** medium. **Reported by:** the performance team.

> We turned on n-gram speculation with k = 4. The code editing service
> got 2.7x faster. The chat service got 6% slower. Is the chat path
> broken?

**Evidence**

- The acceptance rate for each draft token: 0.80 in code editing, 0.15
  in chat.
- A verify step with 4 draft tokens costs about 1.15 times a plain decode
  step at batch 1.
- The chat and the code services run the same engine and the same model.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 2: speculation made the answers less creative

**Severity:** high. The promise of speculation is an exact distribution.
**Reported by:** the evaluation team.

> With speculation on, the sampled answers at temperature 1 repeat the
> prompt more often, and they are less varied.

**Evidence**

- The team built a test: a context where the target model gives
  probability 0.6 to token A and 0.4 to token B. The n-gram draft always
  proposes A.
- In 10,000 samples: without speculation, A 6,003 times. With
  speculation, A 8,412 times.
- The code on a rejection:

  ```python
  accept = random() < min(1, p[draft] / q[draft])
  if not accept:
      token = sample(p)           # sample again from the target
  ```

- The draft is one-hot, so `q[draft] = 1`.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 3: int8 weights that made decode 2.4x slower

**Severity:** medium. **Reported by:** the team that quantized the model.

> Llama-3-8B in bf16 decodes at 104 tokens/s on an A100. With int8
> weights it decodes at 44 tokens/s. Half the bytes gave less than half
> the speed.

**Evidence**

- The quantized linear layer:

  ```python
  def forward(self, x):
      w = self.weight_int8.to(torch.bfloat16) * self.scale     # (out, in)
      return x @ w.t()
  ```

- The batch size is 1.
- The team found that the kernel does not use the int8 tensor cores.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 4: one large weight ruined the layer

**Severity:** high. **Reported by:** the evaluation team.

> After int8 quantization, the KL from bf16 is 0.41 nats. The target is
> 0.015. The [top-1 agreement](../../GLOSSARY.md#top-1-agreement) is 71%.

**Evidence**

- The quantization uses **one** scale for each weight matrix:
  `scale = w.abs().max() / 127`.
- In the worst layer, the largest weight has the value 1.9. The median
  of `|w|` is 0.012. The weights are about normal.
- The team says: "int8 has 256 levels. That must be enough."

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 5: the FP8 cache that works on one model

**Severity:** high. **Reported by:** the team that enabled the FP8 KV
cache.

> With the FP8 KV cache, model A is fine: KL 0.02. Model B is bad: KL
> 0.9, and the answers drift after a few sentences.

**Evidence**

- The FP8 write:

  ```python
  k_fp8 = (k / k_scale).to(torch.float8_e4m3fn)
  ```

  `k_scale` comes from a calibration step. For model B the calibration
  did not run, so `k_scale` is the default, 1.0.
- A debug print for model B: the largest |K| in layers 0 and 1 is 1,150,
  in a few channels.
- For model A, the calibration ran.
- The team thinks that model B is more sensitive to precision.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 6: 3% invalid JSON from the guided mode

**Severity:** medium. The feature is sold as "always valid".
**Reported by:** a customer.

> Your guided JSON mode promises valid JSON. 3.1% of our responses do not
> parse.

**Evidence**

- The invalid responses end like this:

      {"name": "Ada", "skills": ["math", "engines", "poe

- `max_tokens` is 256.
- The customer's schema allows a list of strings of any length.
- The team checked the mask on 10,000 random states of the automaton, and
  it never allows an illegal token.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 7: guided requests make everyone slow

**Severity:** medium. **Reported by:** the performance team.

> When guided requests are in the batch, the time between tokens goes up
> for every request, not only the guided ones.

**Evidence**

- The decode step at batch 16 takes 12 ms without guided requests.
- The measured step, by the number of guided requests in the batch:

  | guided requests | ms for each step |
  |---|---|
  | 0 | 12 |
  | 1 | 50 |
  | 4 | 164 |
  | 8 | 316 |

- For each guided request, the engine builds the mask at each step. It
  tests each of the 151,669 tokens against the automaton, in Python.
- The team says that the GPU is too slow for guided decoding.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 8: a colour that the schema forbids

**Severity:** high. **Reported by:** a customer.

> The schema says `"color": {"enum": ["green", "red", "blue"]}`. We got
> `{"color": "grey"}`.

**Evidence**

- The code that builds the mask:

  ```python
  def allowed(state, token_text):
      return state.accepts(token_text[0])      # the first character
  ```

- The Qwen3 vocabulary has the tokens `green` (13250) and `grey` (34571).
- All the invalid outputs contain one token of several characters where
  the automaton allowed only its first character.
- The team says that the model was fine-tuned on grey elephants.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 9: two GPUs slower than one

**Severity:** low. **Reported by:** the platform team.

> Tensor parallel on 2 L40S cards makes Llama-3-8B 1.76x faster. For
> Qwen3-0.6B it makes decode slower: 2.5 ms for each step against
> 2.2 ms on one card.

**Evidence**

- The two cards connect over PCIe.
- The tensor parallel code does 2 all-reduces in each layer: one after
  the attention and one after the MLP. The values are correct.
- One all-reduce of `hidden x 2` bytes costs about 25 µs.
- The team suspects a bug in the all-reduce for small models.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 10: the rank that got only queries

**Severity:** high. **Reported by:** the team that added tensor
parallel.

> With tp = 1, Qwen3-1.7B is correct. With tp = 2 the output is nonsense
> from the first token.

**Evidence**

- The loader fuses q, k and v into one `qkv` weight, as vLLM does. For
  Qwen3-1.7B its output dimension is 16 x 128 + 8 x 128 + 8 x 128 = 4,096
  rows.
- The sharding:

  ```python
  qkv_shard = qkv_weight.chunk(tp, dim=0)[rank]
  ```

- The MLP uses the same `chunk` and works.
- Each rank prints the right shape: 2,048 rows.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

### Before you open the solution

Go back to each ticket and write one more line: **which piece of evidence was
noise, and why did it look relevant?**